In [1]:
from pathlib import Path
from dataclasses import dataclass
from scipy.stats import chi2
from typing import *

import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np


pd.set_option("display.precision", 10)


ROOT_DIR = Path(os.getcwd()).parent

<h4>Features suggested so far</h4>

<ul>
    <li>1. Buy/sell volume imbalance</li>
    <li>2. Buy/sell number of trades ratio</li>
</ul>

In [2]:
@dataclass
class PumpEvent:
    ticker: str
    pump_time: str

    def __post_init__(self):
        self.pump_time = pd.Timestamp(self.pump_time)


def create_date_range(start: pd.Timestamp, end: pd.Timestamp) -> List[str]:
    """Creates a range of months and years between two dates"""
    start_year = start.year
    start_month = start.month
    end_year = end.year
    end_month = end.month

    date_range = []

    for year in range(start_year, end_year + 1):
        start_month_range = start_month if year == start_year else 1
        end_month_range = end_month if year == end_year else 12

        for month in range(start_month_range, end_month_range + 1):
            date_range.append(pd.Timestamp(year=year, month=month, day=1))

    return date_range


def load_data(pump_event: PumpEvent, lookback_delta: pd.Timedelta) -> pd.DataFrame: 
    rb = pump_event.pump_time
    lb = rb - lookback_delta

    ts_range: List[pd.Timestamp] = create_date_range(start=lb, end=rb)

    # ./data/trades/1INCHBTC
    TICKER_DIR = os.path.join(ROOT_DIR, "data/trades", pump_event.ticker)
    df = pd.DataFrame()

    for date in ts_range:
        month, year = str(date.month).zfill(2), date.year
        # 1INCHBTC-2021-09.parquet
        slug = f"{pump_event.ticker}-{year}-{str(month).zfill(2)}.parquet"
        df_tmp = pd.read_parquet(os.path.join(TICKER_DIR, slug))

        df = pd.concat([df, df_tmp], axis=0)

    df = df.reset_index(drop=True)
    return df

In [3]:
pump = PumpEvent(ticker="NEBLBTC", pump_time="2022-01-02 17:00:00")

df = load_data(pump_event=pump, lookback_delta=pd.Timedelta(days=90))
df.head()

,trade_id,price,qty,time,isBuyerMaker
0,9140960,0.00002498,28.8,2021-10-01 00:00:46.556,True
1,9140961,0.00002498,4.2,2021-10-01 00:00:46.556,True
2,9140962,0.00002502,22.2,2021-10-01 00:01:06.723,False
3,9140963,0.00002504,69.0,2021-10-01 00:01:11.444,False
4,9140964,0.00002505,1.4,2021-10-01 00:01:11.444,False


In [4]:
def create_features(df: pd.DataFrame, window: str) -> pd.DataFrame:
    df["qty_sign"] = df["qty"] * (1 - 2 * df["isBuyerMaker"])

    df_group = df.groupby("time").agg(
        {
            "price": ["first", "min", "mean", "max", "last"],
            "trade_id": ["count"],
            "isBuyerMaker": ["mean"],
            "qty": ["sum"],
            "qty_sign": ["sum"],
        }
    )

    df_group.columns = df_group.columns.map("_".join)

    return df_group

    df_group["time"] = df_group.index

    # ratio of number of trades between buy and sell side
    df_group["buy_sell_num_trades_ratio"] = df_group["isBuyerMaker_mean"].rolling(window).mean()

    df_group["buy_sell_vol_ratio"] = (
        df_group.rolling(window)["qty_sign_sum"].sum() / df_group.rolling(window)["qty_sum"].sum()
    )

    # trade price slippage
    df_group["price_slippage"] = (
        df_group["price_last"] - df_group["price_first"]
    ) / df_group["price_first"]

    df_group["price_slippage"] = df_group["price_slippage"].rolling(window).mean()
    df_group["price_slippage_max"] = df_group["price_slippage"].rolling(window).max()

    # Whale trades stats
    whale_conditon = df_group["qty_sum"] >= df_group["qty_sum"].quantile(.99)

    df_group["whale_trade"] = False
    df_group.loc[whale_conditon, "whale_trade"] = True

    df_group["whale_imbalance"] = (
        (df_group["qty_sign_sum"] * df_group["whale_trade"]).cumsum() / 
        (df_group["qty_sum"] * df_group["whale_trade"]).cumsum()
    ).rolling(window).mean()

    df_group["whale_trading_share"] = (
        (df_group["qty_sum"] * df_group["whale_trade"]).cumsum() / df_group["qty_sum"].cumsum()
    ).rolling(window).mean()

    # ~VWAP
    df_group["vwap"] = (
        (df_group["qty_sum"] * df_group["price_mean"]).cumsum() / df_group["qty_sum"].cumsum()
    ).rolling(window).mean()

    df_group["day_seconds_elapsed"] = (
        df_group["time"] - df_group["time"].dt.floor("D")
    ).dt.total_seconds()

    df_group["sin_time"] = np.sin(2*np.pi*df_group["day_seconds_elapsed"] / 24*60*60)
    df_group["cos_time"] = np.cos(2*np.pi*df_group["day_seconds_elapsed"] / 24*60*60)

    df_group["sin_day"] = np.sin(2*np.pi*df_group["time"].dt.weekday / 7)
    df_group["cos_day"] = np.cos(2*np.pi*df_group["time"].dt.weekday / 7)

    # Chi-squared statistic
    df_group["qty_first_digit"] = (
        df_group["qty_sum"].astype(str).str[0].astype(int)
    )

    df_first_digits = (
        pd.get_dummies(df_group["qty_first_digit"]).rolling(window).sum()
    ).iloc[:, 1:]

    df_first_digits["overall_num_trades"] = df_first_digits.sum(axis=1).rolling(window).sum()

    df_expected_freq = pd.DataFrame()
    # log distribution of first digits Benfard's law
    expected_freq = np.array([np.log10(i + 1) - np.log10(i) for i in range(1, 10)])

    for i in range(1, 10):
        df_expected_freq[i] = expected_freq[i-1] * df_first_digits["overall_num_trades"]

    df_first_digits["chi_squared_stat"] = np.sum(
        np.square(df_first_digits.iloc[:, :-1] - df_expected_freq).div(df_expected_freq, axis=1),
        axis=1
    )

    df_first_digits["pval"] = 1 - chi2.cdf(
        x=df_first_digits["chi_squared_stat"], df=df_first_digits["overall_num_trades"] - 1
    )

    df_group["chi_squared_stat"] = df_first_digits["chi_squared_stat"].rolling(window).mean()
    df_group["pval"] = df_first_digits["pval"].rolling(window).mean()

    return df_group

In [ ]:
df["qty_sign"] = df["qty"] * (1 - 2 * df["isBuyerMaker"])

df_group = df.groupby("time").agg(
    {
        "price": ["first", "min", "mean", "max", "last"],
        "trade_id": ["count"],
        "isBuyerMaker": ["mean"],
        "qty": ["sum"],
        "qty_sign": ["sum"],
    }
)

df_group.columns = df_group.columns.map("_".join)
df_group["time"] = df_group.index

df_group.head(1)

<h4>Whale volume</h4>

$\text{whale volume} = \frac{\text{rolling sum of whale volume}}{\text{rolling overall volume}}$

In [ ]:
is_whale_condition = df_group["qty_sum"] >= df_group["qty_sum"].quantile(.99)

df_group["whale_rolling_vol"] = (
    df_group["qty_sum"] * is_whale_condition
).rolling("1D").sum()

df_group["overall_rolling_vol"] = df_group["qty_sum"].rolling("1D").sum()

df_group["whale_vol_ratio"] = df_group["whale_rolling_vol"] / df_group["overall_rolling_vol"]
df_group["whale_vol_ratio"].plot()
plt.show()

<h4>Whale volume imbalance, buys against sells in whale trades</h4>

In [ ]:
df_group["whale_net_position"] = (df_group["qty_sign_sum"] * is_whale_condition).rolling("1D").sum()

df_group["whale_trade_imbalance"] = (
    df_group["whale_net_position"] / df_group["whale_rolling_vol"]
)

df_group["whale_trade_imbalance"].plot()
plt.show()

<h4>Whale trading volume in time</h4>

In [ ]:
df_group["weekday_trade"] = df_group["time"].dt.weekday
df_group["weekday_trade"]

In [ ]:
df_whale_days = pd.get_dummies(
    df_group[is_whale_condition]["weekday_trade"]
).rolling("14D").sum()

whale_num_trades = df_group[is_whale_condition].rolling("14D")["weekday_trade"].count()

df_whale_days = df_whale_days.div(whale_num_trades, axis=0)
df_whale_days.plot.area()

plt.title("Whale trades day of the week")
plt.axvline(x=pump.pump_time, color="white", linestyle="--")
plt.show()

In [ ]:
# adjusted for volume

df_whale_day_volume = (
    pd.get_dummies(df_group[is_whale_condition]["weekday_trade"]).mul(df_group[is_whale_condition]["qty_sum"], axis=0)
)

df_whale_day_vol_ratio = df_whale_day_volume.rolling("14D").sum().div(
    df_group[is_whale_condition]["qty_sum"].rolling("14D").sum(), axis=0
)

df_whale_day_vol_ratio.plot.area()
plt.show()

<h4>Whale trade hours</h4>

In [ ]:
df_group["trade_hour"] = df_group["time"].dt.hour

df_whale_days = (
    pd.get_dummies(df_group[is_whale_condition]["trade_hour"]).rolling("14D").sum()
)

whale_num_trades = df_group[is_whale_condition].rolling("14D")["trade_hour"].count()

df_whale_days = df_whale_days.div(whale_num_trades, axis=0)
df_whale_days.plot.area(figsize=(16, 10))

plt.title("Whale trades day of the week")
plt.axvline(x=pump.pump_time, color="white", linestyle="--")
plt.show()

In [ ]:
df_whale_day_volume = pd.get_dummies(df_group[is_whale_condition]["trade_hour"]).mul(
    df_group[is_whale_condition]["qty_sum"], axis=0
)

df_whale_day_vol_ratio = (
    df_whale_day_volume.rolling("14D")
    .sum()
    .div(df_group[is_whale_condition]["qty_sum"].rolling("14D").sum(), axis=0)
)

df_whale_day_vol_ratio.plot.area(figsize=(16, 10))
plt.show()

<h4>Whale Buy trading volume by day</h4>

In [ ]:
buy_side_condition = df_group["qty_sign_sum"] > 0

df_whale_days = (
    pd.get_dummies(df_group[is_whale_condition & buy_side_condition]["weekday_trade"]).rolling("14D").sum()
)

whale_num_trades = df_group[is_whale_condition & buy_side_condition].rolling("14D")["weekday_trade"].count()

df_whale_days = df_whale_days.div(whale_num_trades, axis=0)
df_whale_days.plot.area()

plt.title("Whale trades day of the week")
plt.axvline(x=pump.pump_time, color="white", linestyle="--")
plt.show()